# Group 13 · Llama-Based Customer-Service Chatbot (No RAG)
**MSBA 610 Advanced Text Analytics**

This notebook runs the whole project on Google Colab, so it no longer depends on one laptop.
It loads the Bitext customer-support dataset from Hugging Face, cleans and explores it, builds text
representations, trains two intent-router models, and compares Llama-3.2-3B-Instruct before and after
QLoRA fine-tuning. At the end it downloads **`group13_deploy.zip`**, which the Streamlit app reads.

**Pipeline**
1. Data cleaning with regular expressions → 2. Preprocessing → 3. Exploratory text analytics →
4. Text representation → 5. Intent router: Model 1 (Naive Bayes) vs Model 2 (Linear SVM) →
6. Llama generator: base vs QLoRA fine-tuned → 7. Evaluation and comparison → 8. Export for Streamlit

### Before you run
1. **Runtime → Change runtime type → T4 GPU**, then **Save**.
2. Llama is gated. On huggingface.co, open `meta-llama/Llama-3.2-3B-Instruct` and accept the licence.
   Then create a **Read** token (Settings → Access Tokens).
3. In Colab, click the **key icon** (Secrets) in the left bar. Add a secret named `HF_TOKEN` with the
   token as its value, and switch on **Notebook access**.
   *If the licence request is still pending, the notebook falls back to an open copy of the same model.*
4. **Runtime → Run all.** Allow Google Drive access when asked.

**Time on a free T4:** about 45–70 minutes. Progress is saved to Google Drive (`MyDrive/group13_chatbot`).
If Colab disconnects, just **Run all** again: finished steps (base answers, the fine-tuned adapter,
fine-tuned answers) are loaded from Drive instead of being redone.

## 0 · Setup

In [ ]:
!pip -q install peft bitsandbytes rouge_score bert_score sentence-transformers

In [ ]:
import os, json, math, random, time, re, shutil, warnings, platform, datetime
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# ---------------- Experiment configuration (change here only) ----------------
SEED = 42
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"        # official, gated
FALLBACK_MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"  # open copy of the same weights

N_TRAIN = 2000            # fine-tuning examples (stratified by intent)
N_VAL = 200               # validation examples (reference perplexity)
N_TEST_PER_INTENT = 6     # 27 intents x 6 = 162 held-out test questions
MAX_LEN = 512             # max tokens per training example
MAX_NEW_TOKENS = 300      # max answer length at generation
GEN_BATCH = 16            # answers generated in parallel

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR, EPOCHS, BATCH, GRAD_ACCUM = 2e-4, 1, 4, 4   # effective batch = 16

# The intent router's prediction is passed to Llama as a routing note,
# so the router actually feeds the generator (as in our architecture).
USE_INTENT_HINT = True

SYSTEM_PROMPT = (
    "You are a helpful, concise customer-service assistant for an e-commerce company. "
    "Answer only using information the customer provides in their message. "
    "Never invent order numbers, dates, prices, phone numbers, links or account details; "
    "if a specific detail is needed, refer to it with a placeholder such as {{Order Number}}."
)

random.seed(SEED); np.random.seed(SEED)

# ---------------- Storage: Google Drive so work survives a disconnect ----------------
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/group13_chatbot"
except Exception as e:
    print("Drive not mounted, saving locally only:", e)
    WORK_DIR = "/content/group13_chatbot"
ART_DIR = os.path.join(WORK_DIR, "artifacts")
ADAPTER_DIR = os.path.join(WORK_DIR, "lora_adapter")
CACHE_DIR = os.path.join(WORK_DIR, "cache")
for d in (ART_DIR, CACHE_DIR):
    os.makedirs(d, exist_ok=True)

def save_json(obj, name):
    with open(os.path.join(ART_DIR, name), "w") as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

# ---------------- Hugging Face login ----------------
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Logged in to Hugging Face.")
else:
    print("No HF_TOKEN secret found - will use the open fallback model.")
print("Working folder:", WORK_DIR)

### Shared cleaning module
The next cell writes `textprep.py`. **The same file is in the Streamlit repo**, so the app cleans a typed
message exactly the way the training data was cleaned.

In [ ]:
%%writefile textprep.py
"""
textprep.py
===========
Group 13 - shared text-cleaning and preprocessing functions.

This ONE file is used by both the Colab notebook (training) and the Streamlit
app (live intent routing). Keeping it in one place guarantees that a message
typed into the app is cleaned exactly the same way as the training data was,
which fixes the train/serve mismatch in the earlier version (the app used to
send raw lower-case text to a model trained on stemmed, stopword-free text).

Two cleaning tracks are used on purpose:

  * light_clean()  -> for the Llama generator. Keeps case and punctuation,
                      because a generative model must learn natural writing.
  * router_prep()  -> for the classical intent router (TF-IDF + NB / SVM).
                      Regex clean -> lowercase -> stopword removal -> stemming.
"""

import re
import unicodedata

from nltk.stem import PorterStemmer

# ---------------------------------------------------------------------------
# Regular expressions (Stage 1: data cleaning)
# ---------------------------------------------------------------------------
RE_URL = re.compile(r"https?://\S+|www\.\S+")
RE_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
RE_MULTISPACE = re.compile(r"\s+")
RE_NON_ALNUM = re.compile(r"[^a-zA-Z0-9{}\s]")       # keep {{ }} slot braces
RE_REPEATED_CH = re.compile(r"([a-zA-Z])\1{2,}")      # soooo -> soo
RE_SLOT = re.compile(r"\{\{\s*(.*?)\s*\}\}")          # {{ Order Number }}
RE_SLOT_TOKEN = re.compile(r"__slot\d+__")

# Named rules, so the notebook can report how often each one fires.
REGEX_RULES = {
    "url": RE_URL,
    "email": RE_EMAIL,
    "slot_placeholder": RE_SLOT,
    "elongated_word": RE_REPEATED_CH,
    "punctuation_symbol": RE_NON_ALNUM,
    "extra_whitespace": re.compile(r"\s{2,}|^\s|\s$"),
}

# Bitext answers use {{Slot}} placeholders such as {{Order Number}}. They are
# KEPT, not removed: the chatbot should learn to write a placeholder instead of
# inventing an order number (this is how a No-RAG bot avoids hallucinating).


def extract_slots(text):
    """Return slot names found in a string, e.g. ['Order Number']."""
    return [s.strip() for s in RE_SLOT.findall(str(text))]


def normalise_slots(text):
    """'{{ Order  Number }}' -> '{{Order Number}}'."""
    return RE_SLOT.sub(lambda m: "{{" + RE_MULTISPACE.sub(" ", m.group(1)).strip() + "}}", text)


def light_clean(text):
    """Track A - for the Llama generator. Unicode + whitespace + slot spacing only."""
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = normalise_slots(text)
    # tidy spaces but keep line breaks, which Bitext answers use for steps
    text = "\n".join(RE_MULTISPACE.sub(" ", line).strip() for line in text.splitlines())
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def regex_clean(text):
    """Track B, step 1 - rule-based cleaning for the router."""
    text = light_clean(text)
    text = RE_URL.sub(" urltoken ", text)
    text = RE_EMAIL.sub(" emailtoken ", text)
    text = RE_REPEATED_CH.sub(r"\1\1", text)
    # expand negative contractions so the apostrophe stripper keeps "not"
    text = re.sub(r"(?i)\bwon't\b", "will not", text)
    text = re.sub(r"(?i)\bcan't\b", "can not", text)
    text = re.sub(r"(?i)n't\b", " not", text)
    # protect {{slots}} from the punctuation stripper with letters-only markers
    slots = []

    def _hold(m):
        slots.append(m.group(1))
        return " slotmarker%s " % chr(97 + len(slots) - 1)

    text = RE_SLOT.sub(_hold, text)
    text = RE_NON_ALNUM.sub(" ", text)
    for i, s in enumerate(slots):
        text = text.replace("slotmarker%s" % chr(97 + i), "{{" + s.replace(" ", "_") + "}}")
    return RE_MULTISPACE.sub(" ", text).strip()


# ---------------------------------------------------------------------------
# Stage 2: preprocessing (tokenise, lowercase, stopwords, stemming)
# ---------------------------------------------------------------------------
# Curated English stopword list. Negation words (not, no, don't, can't ...) are
# deliberately NOT removed: "I don't want to cancel" and "I want to cancel"
# should not look identical to the router.
NEGATIONS = {
    "no", "nor", "not", "don't", "doesn't", "didn't", "can't", "cannot",
    "couldn't", "won't", "wouldn't", "shouldn't", "isn't", "aren't", "wasn't",
    "weren't", "haven't", "hasn't", "hadn't", "mustn't", "shan't",
}

STOPWORDS = set("""
a about above after again against all am an and any are as at be because been
before being below between both but by could did do does doing down during each
few for from further had has have having he he'd he'll he's her here here's
hers herself him himself his how how's i i'd i'll i'm i've if in into is it
it's its itself let's me more most my myself of off on once only or other ought
our ours ourselves out over own same she she'd she'll she's should so some such
than that that's the their theirs them themselves then there there's these they
they'd they'll they're they've this those through to too under until up very
was we we'd we'll we're we've were what what's when when's where where's which
while who who's whom why why's with would you you'd you'll you're you've your
yours yourself yourselves
""".split()) - NEGATIONS

_stemmer = PorterStemmer()


def preprocess(text, remove_stopwords=True, stem=True):
    tokens = []
    for tok in text.lower().split():
        if tok.startswith("{{"):
            tokens.append("slottoken")
            continue
        if remove_stopwords and tok in STOPWORDS:
            continue
        tokens.append(_stemmer.stem(tok) if stem else tok)
    return " ".join(tokens)


def router_text(text):
    """Full Track B pipeline for one message."""
    return preprocess(regex_clean(text))


def router_prep(texts):
    """Vectorised version used inside the saved sklearn Pipeline."""
    return [router_text(t) for t in texts]


# ---------------------------------------------------------------------------
# Hallucination check (used by the notebook's evaluation)
# ---------------------------------------------------------------------------
# A No-RAG bot has no access to the customer's account, so any concrete detail
# it states (an order number, a phone number, a price, a link, a policy time
# window) that the customer did not supply is invented.
SPECIFIC_PATTERNS = {
    "url": re.compile(r"https?://\S+|www\.\S+"),
    "email": re.compile(r"\b[\w.+-]+@[\w-]+\.[a-z]{2,}\b", re.I),
    "phone": re.compile(r"(?<!\w)\+?\d[\d\-\s().]{7,}\d(?!\w)"),
    "long_number": re.compile(r"(?<![\w{])\d{4,}(?![\w}])"),
    "money": re.compile(r"[$€£]\s?\d[\d,]*(?:\.\d+)?|\b\d[\d,]*(?:\.\d+)?\s?(?:usd|eur|gbp|dollars)\b", re.I),
    "time_window": re.compile(r"\b\d+\s?(?:-|to)?\s?\d*\s?(?:business\s|working\s)?(?:days?|hours?|weeks?|minutes?)\b", re.I),
}


def invented_specifics(answer, question):
    """Return {type: [values]} for concrete details in `answer` that do not
    appear in `question`. {{slot}} placeholders are removed first, because a
    placeholder is the correct way NOT to invent a value."""
    answer_wo_slots = RE_SLOT.sub(" ", str(answer))
    q = str(question).lower()
    q_digits = re.sub(r"\D", "", q)
    found = {}
    for kind, pat in SPECIFIC_PATTERNS.items():
        for val in pat.findall(answer_wo_slots):
            val = val.strip().rstrip(".,;:!?)")
            v = val.lower()
            digits = re.sub(r"\D", "", v)
            grounded = v in q or (len(digits) >= 4 and digits in q_digits)
            if not grounded:
                found.setdefault(kind, []).append(val)
        # blank out what this pattern matched so e.g. a phone number is not
        # counted again as a "long number"
        answer_wo_slots = pat.sub(" ", answer_wo_slots)
    return found

In [ ]:
import importlib, sys
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import textprep
importlib.reload(textprep)
print(textprep.router_text("I don't want to CANCEL order {{ Order  Number }} pleaseeee!!!"))

## 1 · Load the data
Loaded directly from Hugging Face, so no local file is needed.

In [ ]:
from datasets import load_dataset
raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset", split="train").to_pandas()
print(raw.shape)
raw.head()

## 2 · Data cleaning (regular expressions) and preprocessing
Two cleaning tracks:
* **Track A – light clean (for Llama):** unicode, quotes, whitespace and `{{slot}}` spacing only. Case and
  punctuation stay, because a generative model has to learn natural writing.
* **Track B – router text (for Naive Bayes / SVM):** Track A, plus URL/e-mail tokens, elongated-word repair,
  punctuation removal, lower-casing, stopword removal (negations kept) and Porter stemming.

`{{Slot}}` placeholders are kept. A No-RAG bot cannot see the customer's account, so the correct behaviour
is to write `{{Order Number}}` rather than make one up.

In [ ]:
df = raw.copy()
n_raw = len(df)

# How often each regex rule fires on the raw text
rule_hits = []
for col in ["instruction", "response"]:
    s = df[col].astype(str)
    for name, pat in textprep.REGEX_RULES.items():
        hit = s.map(lambda t: bool(pat.search(t)))  # Python's re engine (supports back-references)
        rule_hits.append({"field": col, "rule": name,
                          "rows_matched": int(hit.sum()), "pct_rows": round(100 * hit.mean(), 2)})
rule_hits = pd.DataFrame(rule_hits)
rule_hits.to_csv(os.path.join(ART_DIR, "regex_rule_hits.csv"), index=False)

df = df.dropna(subset=["instruction", "response", "intent", "category"])
n_after_na = len(df)
df["instruction_clean"] = df["instruction"].map(textprep.light_clean)
df["response_clean"] = df["response"].map(textprep.light_clean)
df = df[(df.instruction_clean.str.len() > 0) & (df.response_clean.str.len() > 0)]
n_after_empty = len(df)

# Exact duplicate question/answer pairs
df = df.drop_duplicates(subset=["instruction_clean", "response_clean"])
n_after_pair_dedup = len(df)

# Same question appearing more than once (even with a different answer).
# Kept only once, so an identical question can never be in both train and test.
key = df.instruction_clean.str.lower()
conflicting = df.assign(k=key).groupby("k")["intent"].nunique()
n_conflicting_questions = int((conflicting > 1).sum())
df = df.loc[~key.duplicated()].reset_index(drop=True)
n_final = len(df)

df["router_text"] = df["instruction_clean"].map(textprep.router_text)
df["instr_words"] = df.instruction_clean.str.split().map(len)
df["resp_words"] = df.response_clean.str.split().map(len)
df["instr_slots"] = df.instruction_clean.map(textprep.extract_slots)
df["resp_slots"] = df.response_clean.map(textprep.extract_slots)

data_summary = {
    "rows_raw": n_raw, "rows_after_missing": n_after_na, "rows_after_empty": n_after_empty,
    "rows_after_pair_dedup": n_after_pair_dedup, "rows_final": n_final,
    "duplicate_pairs_removed": n_after_empty - n_after_pair_dedup,
    "repeated_questions_removed": n_after_pair_dedup - n_final,
    "questions_with_conflicting_intents": n_conflicting_questions,
    "n_intents": int(df.intent.nunique()), "n_categories": int(df.category.nunique()),
}
save_json(data_summary, "data_summary.json")

# Before/after examples (prefer rows where cleaning visibly changes something)
changed = df[df.instruction != df.instruction_clean]
ex_pool = pd.concat([changed.sample(min(10, len(changed)), random_state=SEED),
                     df[df.instr_slots.map(len) > 0].sample(min(8, (df.instr_slots.map(len) > 0).sum()), random_state=SEED),
                     df.sample(12, random_state=SEED)]).drop_duplicates("instruction").head(25)
ex_pool[["intent", "instruction", "instruction_clean", "router_text"]].rename(columns={
    "instruction": "raw", "instruction_clean": "track_a_light_clean", "router_text": "track_b_router_text"
}).to_csv(os.path.join(ART_DIR, "cleaning_examples.csv"), index=False)

print(json.dumps(data_summary, indent=2))
rule_hits

### Train / validation / test split
One stratified split (80/10/10 by intent) is made once and reused by every model, so no test question is
ever seen in training.

In [ ]:
from sklearn.model_selection import train_test_split
idx_train, idx_tmp = train_test_split(df.index, test_size=0.2, random_state=SEED, stratify=df.intent)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, random_state=SEED, stratify=df.loc[idx_tmp, "intent"])
df["split"] = "train"
df.loc[idx_val, "split"] = "val"
df.loc[idx_test, "split"] = "test"
df.to_csv(os.path.join(CACHE_DIR, "clean_with_split.csv"), index=False)
df.split.value_counts()

## 3 · Exploratory text analytics

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def top_ngrams(texts, ngram, k=25):
    cv = CountVectorizer(ngram_range=(ngram, ngram), min_df=2, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
    m = cv.fit_transform(texts)
    counts = np.asarray(m.sum(0)).ravel()
    order = counts.argsort()[::-1][:k]
    vocab = cv.get_feature_names_out()
    return [{"term": vocab[i], "count": int(counts[i])} for i in order]

def length_stats(s):
    return {"mean": round(float(s.mean()), 1), "median": float(s.median()),
            "p95": float(s.quantile(0.95)), "max": int(s.max())}

# Bitext "flags" mark the language variation in each question (letters per the dataset card)
FLAG_NAMES = {"B": "Basic syntax", "S": "Syntactic variation", "L": "Lexical variation", "M": "Morphological variation",
              "I": "Interrogative", "C": "Coordinated / complex", "P": "Politeness", "Q": "Colloquial",
              "W": "Offensive language", "K": "Keyword mode", "E": "Abbreviations", "Z": "Errors and typos",
              "N": "Negation"}
flag_counts = {}
if "flags" in df.columns:
    for f in df["flags"].fillna(""):
        for ch in set(str(f)):
            if ch.isalpha():
                flag_counts[ch] = flag_counts.get(ch, 0) + 1

intent_cat = df.groupby("intent")["category"].agg(lambda s: s.mode()[0])
slot_resp = pd.Series([s for l in df.resp_slots for s in l]).value_counts().head(20)
slot_instr = pd.Series([s for l in df.instr_slots for s in l]).value_counts().head(20)

per_cat_tokens = {}
for cat, g in df.groupby("category"):
    per_cat_tokens[cat] = [t["term"] for t in top_ngrams(g.router_text, 1, 10)]

eda = {
    "category_counts": df.category.value_counts().to_dict(),
    "intent_counts": [{"intent": i, "category": intent_cat[i], "count": int(c)} for i, c in df.intent.value_counts().items()],
    "flag_counts": [{"flag": k, "meaning": FLAG_NAMES.get(k, k), "count": v} for k, v in sorted(flag_counts.items(), key=lambda x: -x[1])],
    "instruction_words": length_stats(df.instr_words),
    "response_words": length_stats(df.resp_words),
    "pct_responses_with_slots": round(100 * (df.resp_slots.map(len) > 0).mean(), 1),
    "pct_instructions_with_slots": round(100 * (df.instr_slots.map(len) > 0).mean(), 1),
    "top_response_slots": [{"slot": k, "count": int(v)} for k, v in slot_resp.items()],
    "top_instruction_slots": [{"slot": k, "count": int(v)} for k, v in slot_instr.items()],
    "top_unigrams": top_ngrams(df.router_text, 1),
    "top_bigrams": top_ngrams(df.router_text, 2),
    "top_tokens_per_category": per_cat_tokens,
}
save_json(eda, "eda_stats.json")
df.sample(min(5000, len(df)), random_state=SEED)[["category", "intent", "instr_words", "resp_words"]] \
  .to_csv(os.path.join(ART_DIR, "length_sample.csv"), index=False)

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
df.category.value_counts().plot.barh(ax=ax[0], title="Questions per category")
ax[1].hist(df.instr_words, bins=40, alpha=.7, label="question"); ax[1].hist(df.resp_words.clip(upper=400), bins=40, alpha=.7, label="answer")
ax[1].set_title("Length in words"); ax[1].legend(); plt.tight_layout(); plt.show()
print("Answer length:", eda["response_words"], "| answers using slots:", eda["pct_responses_with_slots"], "%")

## 4 · Text representation
* **TF-IDF + LSA (Truncated SVD, 128-d):** sparse, interpretable features and a dense word-space built from them.
* **Sentence embeddings (all-MiniLM-L6-v2):** pretrained contextual embeddings. They are compared with LSA on
  how well they separate the intents (silhouette score), and used later for the relevance metrics.
* **Llama tokenizer:** sub-word token lengths, which set `MAX_LEN` and `MAX_NEW_TOKENS`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

train_df = df[df.split == "train"]
tfidf_lsa = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)
X_lsa_tfidf = tfidf_lsa.fit_transform(train_df.router_text)
svd = TruncatedSVD(n_components=128, random_state=SEED).fit(X_lsa_tfidf)

# Word vectors for the most frequent unigrams (for the "nearest words" explorer in the app)
vocab = tfidf_lsa.get_feature_names_out()
doc_freq = np.asarray((X_lsa_tfidf > 0).sum(0)).ravel()
uni = np.array([(" " not in v) for v in vocab])
keep = np.where(uni)[0][np.argsort(-doc_freq[uni])][:4000]
word_vecs = svd.components_.T[keep]
word_vecs = word_vecs / (np.linalg.norm(word_vecs, axis=1, keepdims=True) + 1e-9)
np.savez_compressed(os.path.join(ART_DIR, "lsa_word_vectors.npz"), vocab=vocab[keep], vectors=word_vecs.astype(np.float16))

def nearest(word, k=8):
    w = textprep.preprocess(word)
    hits = np.where(vocab[keep] == w)[0]
    if not len(hits): return []
    sims = word_vecs @ word_vecs[hits[0]]
    return [(vocab[keep][i], round(float(sims[i]), 3)) for i in np.argsort(-sims)[1:k + 1]]
for w in ["refund", "cancel", "password", "invoice", "delivery"]:
    print(w, "->", nearest(w, 6))

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
import torch

st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")
map_df = df.sample(3000, random_state=SEED).reset_index(drop=True)
E_st = st_model.encode(map_df.instruction_clean.tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
E_lsa = svd.transform(tfidf_lsa.transform(map_df.router_text))

rep_compare = {
    "lsa_explained_variance": round(float(svd.explained_variance_ratio_.sum()), 4),
    "lsa_dims": 128, "minilm_dims": int(E_st.shape[1]),
    "silhouette_intent_lsa": round(float(silhouette_score(E_lsa, map_df.intent, metric="cosine")), 4),
    "silhouette_intent_minilm": round(float(silhouette_score(E_st, map_df.intent, metric="cosine")), 4),
    "silhouette_category_lsa": round(float(silhouette_score(E_lsa, map_df.category, metric="cosine")), 4),
    "silhouette_category_minilm": round(float(silhouette_score(E_st, map_df.category, metric="cosine")), 4),
}
xy = TSNE(n_components=2, random_state=SEED, init="pca", perplexity=35).fit_transform(E_st)
map_df.assign(x=xy[:, 0].round(3), y=xy[:, 1].round(3))[["x", "y", "category", "intent", "instruction_clean"]] \
      .rename(columns={"instruction_clean": "question"}).to_csv(os.path.join(ART_DIR, "embedding_map.csv"), index=False)
print(json.dumps(rep_compare, indent=2))

In [ ]:
from transformers import AutoTokenizer

def resolve_model_id():
    for mid in [MODEL_ID, FALLBACK_MODEL_ID]:
        try:
            AutoTokenizer.from_pretrained(mid)
            return mid
        except Exception as e:
            print(f"Could not open {mid}: {type(e).__name__}. Trying next option.")
    raise RuntimeError("No Llama model could be loaded - check internet / HF token.")

MODEL_USED = resolve_model_id()
print("Using model:", MODEL_USED)
tok = AutoTokenizer.from_pretrained(MODEL_USED)
EOT = "<|eot_id|>"
EOT_ID = tok.convert_tokens_to_ids(EOT)
if tok.pad_token is None:
    tok.pad_token = "<|finetune_right_pad_id|>" if "<|finetune_right_pad_id|>" in tok.get_vocab() else tok.eos_token

INTENT2CAT = intent_cat.to_dict()

def build_messages(question, intent=None):
    system = SYSTEM_PROMPT
    if USE_INTENT_HINT and intent:
        system += f"\nRouting note: the intent router classified this message as '{intent}' (category: {INTENT2CAT.get(intent, 'unknown')})."
    return [{"role": "system", "content": system}, {"role": "user", "content": question}]

def prompt_text(question, intent=None):
    return tok.apply_chat_template(build_messages(question, intent), add_generation_prompt=True, tokenize=False)

tk_sample = df.sample(3000, random_state=SEED)
q_tok = np.array([len(tok(q, add_special_tokens=False)["input_ids"]) for q in tk_sample.instruction_clean])
a_tok = np.array([len(tok(a, add_special_tokens=False)["input_ids"]) for a in tk_sample.response_clean])
p_tok = np.array([len(tok(prompt_text(q, i), add_special_tokens=False)["input_ids"]) for q, i in zip(tk_sample.instruction_clean[:300], tk_sample.intent[:300])])
full = p_tok.mean() + a_tok
slot_example = "Your order {{Order Number}} has been cancelled."
token_stats = {
    "model": MODEL_USED,
    "question_tokens": length_stats(pd.Series(q_tok)),
    "answer_tokens": length_stats(pd.Series(a_tok)),
    "prompt_tokens_with_system": length_stats(pd.Series(p_tok)),
    "tokens_per_word_answer": round(float(a_tok.sum() / tk_sample.resp_words.sum()), 3),
    "pct_answers_over_max_new_tokens": round(100 * float((a_tok > MAX_NEW_TOKENS).mean()), 2),
    "pct_examples_over_max_len": round(100 * float((full > MAX_LEN).mean()), 2),
    "max_len": MAX_LEN, "max_new_tokens": MAX_NEW_TOKENS,
    "slot_tokenisation_example": {"text": slot_example,
                                  "tokens": tok.convert_ids_to_tokens(tok(slot_example, add_special_tokens=False)["input_ids"])},
}
rep_compare["llama_tokens"] = token_stats
save_json(rep_compare, "representation.json")
print(json.dumps(token_stats, indent=2))

## 5 · Intent router: Model 1 (Multinomial Naive Bayes) vs Model 2 (Linear SVM)
Both use TF-IDF (unigrams + bigrams) on Track B text. The router's job is to identify what the customer
wants, and its prediction is passed to Llama as a routing note. Each model is saved as one sklearn
Pipeline that **includes the cleaning step**, so the app gives it raw text, exactly as in training.

In [ ]:
import pickle, sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score

tr = df[df.split == "train"]; te = df[df.split == "test"]
labels = sorted(df.intent.unique())

def make_model(kind):
    clf = MultinomialNB(alpha=0.3) if kind == "nb" else LinearSVC(C=1.0, class_weight="balanced", random_state=SEED)
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=20000, sublinear_tf=True)),
                     ("clf", clf)])

ROUTERS = {"nb": "Model 1 - Multinomial Naive Bayes", "svm": "Model 2 - Linear SVM"}
router_rows, cv_rows, cm_rows, per_intent, fitted = [], [], [], [], {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for key, name in ROUTERS.items():
    cv = cross_val_score(make_model(key), tr.router_text, tr.intent, cv=skf, scoring="f1_macro", n_jobs=-1)
    t0 = time.time(); m = make_model(key).fit(tr.router_text, tr.intent); fit_s = time.time() - t0
    pred = m.predict(te.router_text)
    p, r, f, _ = precision_recall_fscore_support(te.intent, pred, average="macro", zero_division=0)
    pw, rw, fw, _ = precision_recall_fscore_support(te.intent, pred, average="weighted", zero_division=0)
    router_rows.append({"key": key, "model": name, "accuracy": accuracy_score(te.intent, pred),
                        "precision_macro": p, "recall_macro": r, "f1_macro": f,
                        "precision_weighted": pw, "recall_weighted": rw, "f1_weighted": fw,
                        "cv_f1_macro_mean": cv.mean(), "cv_f1_macro_std": cv.std(), "fit_seconds": fit_s})
    cv_rows += [{"key": key, "fold": i + 1, "f1_macro": s} for i, s in enumerate(cv)]
    cm = confusion_matrix(te.intent, pred, labels=labels)
    for i, a in enumerate(labels):
        for j, b in enumerate(labels):
            if cm[i, j]:
                cm_rows.append({"key": key, "true": a, "pred": b, "count": int(cm[i, j]), "rate": cm[i, j] / cm[i].sum()})
    f_int = f1_score(te.intent, pred, labels=labels, average=None, zero_division=0)
    per_intent += [{"key": key, "intent": l, "category": INTENT2CAT[l], "f1": v} for l, v in zip(labels, f_int)]
    fitted[key] = (m, pred)

router_df = pd.DataFrame(router_rows)
router_df.to_csv(os.path.join(ART_DIR, "router_comparison.csv"), index=False)
pd.DataFrame(cv_rows).to_csv(os.path.join(ART_DIR, "router_cv.csv"), index=False)
pd.DataFrame(cm_rows).to_csv(os.path.join(ART_DIR, "router_confusion.csv"), index=False)
pd.DataFrame(per_intent).to_csv(os.path.join(ART_DIR, "router_per_intent.csv"), index=False)
BEST_ROUTER = router_df.sort_values("f1_macro", ascending=False).iloc[0]["key"]

# Interpretation: the tokens that push each intent up the most
top_tokens = {}
for key, (m, _) in fitted.items():
    feats = m.named_steps["tfidf"].get_feature_names_out()
    clf = m.named_steps["clf"]
    if key == "svm":
        weights = clf.coef_
    else:  # NB: how much more likely a token is in this intent than in the others
        lp = clf.feature_log_prob_
        weights = lp - (lp.sum(0, keepdims=True) - lp) / (lp.shape[0] - 1)
    top_tokens[key] = {c: feats[np.argsort(weights[i])[::-1][:12]].tolist() for i, c in enumerate(clf.classes_)}
save_json(top_tokens, "router_top_tokens.json")

# Mistakes of the best router (for error analysis)
bm, bpred = fitted[BEST_ROUTER]
errs = te.assign(predicted=bpred)[lambda d: d.intent != d.predicted]
errs[["instruction_clean", "intent", "predicted"]].rename(columns={"instruction_clean": "question", "intent": "true_intent"}) \
    .head(40).to_csv(os.path.join(ART_DIR, "router_errors.csv"), index=False)

# Save deployable pipelines: raw text -> cleaning -> TF-IDF -> classifier
prep = FunctionTransformer(textprep.router_prep)
prep.fit(["warm up"])
for key, (m, _) in fitted.items():
    deploy = Pipeline([("prep", prep), ("tfidf", m.named_steps["tfidf"]), ("clf", m.named_steps["clf"])])
    assert (deploy.predict(te.instruction_clean[:50]) == m.predict(te.router_text[:50])).all()
    with open(os.path.join(ART_DIR, f"router_{key}.pkl"), "wb") as f:
        pickle.dump(deploy, f)

print("Best router:", ROUTERS[BEST_ROUTER])
router_df.round(4)

## 6 · Llama generator: base vs QLoRA fine-tuned
**Model A (base):** Llama-3.2-3B-Instruct, zero-shot, same system prompt.
**Model B (fine-tuned):** the same model with LoRA adapters trained on Bitext question/answer pairs (QLoRA:
4-bit base weights, about 1% of parameters trained). No retrieval is used at any point (No-RAG).

Training uses the true intent in the routing note. At test time, both models get the intent predicted
by the best router, as the deployed system would.

In [ ]:
def stratified_sample(frame, n, seed=SEED):
    frac = min(1.0, (n + 30) / len(frame))
    s = frame.groupby("intent").sample(frac=frac, random_state=seed)
    return s.sample(min(n, len(s)), random_state=seed)

llm_train = stratified_sample(df[df.split == "train"], N_TRAIN)
llm_val = stratified_sample(df[df.split == "val"], N_VAL)
llm_test = df[df.split == "test"].groupby("intent").sample(n=N_TEST_PER_INTENT, random_state=SEED).reset_index(drop=True)
llm_test["predicted_intent"] = fitted[BEST_ROUTER][0].predict(llm_test.router_text)
print(len(llm_train), "train |", len(llm_val), "val |", len(llm_test), "test",
      "| router accuracy on these test questions:", round((llm_test.predicted_intent == llm_test.intent).mean(), 3))

In [ ]:
import torch, transformers
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from packaging import version
assert torch.cuda.is_available(), "No GPU. Go to Runtime > Change runtime type > T4 GPU, then Run all again."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME, "| transformers", transformers.__version__)

def load_base_model():
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    dtype_kw = {"dtype": torch.float16} if version.parse(transformers.__version__) >= version.parse("4.56") else {"torch_dtype": torch.float16}
    m = AutoModelForCausalLM.from_pretrained(MODEL_USED, quantization_config=bnb, device_map={"": 0}, **dtype_kw)
    m.config.use_cache = True
    return m

def encode_example(question, answer, intent):
    p_ids = tok(prompt_text(question, intent), add_special_tokens=False)["input_ids"]
    a_ids = tok(answer + EOT, add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return {"input_ids": ids, "labels": labels}

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        ids[i, :n] = torch.tensor(b["input_ids"]); lab[i, :n] = torch.tensor(b["labels"]); att[i, :n] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": att}

@torch.no_grad()
def reference_perplexity(model, frame, bs=4):
    """Perplexity of the gold Bitext answers under the model (lower = closer to the target style)."""
    model.eval()
    enc = [encode_example(q, a, i) for q, a, i in zip(frame.instruction_clean, frame.response_clean, frame.intent)]
    nll, ntok = 0.0, 0
    for s in range(0, len(enc), bs):
        b = {k: v.to("cuda") for k, v in collate(enc[s:s + bs]).items()}
        out = model(**b)
        n = int((b["labels"][:, 1:] != -100).sum())
        nll += float(out.loss) * n; ntok += n
    return math.exp(nll / ntok)

@torch.no_grad()
def generate_answers(model, frame, cache_name):
    """Greedy answers for every test question; saved to Drive after each batch so a disconnect loses little."""
    path = os.path.join(CACHE_DIR, cache_name)
    done = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame(columns=["row", "answer", "new_tokens", "hit_limit"])
    finished = set(int(r) for r in done.row)
    todo = [int(i) for i in np.argsort(frame.instruction_clean.str.len().values) if int(i) not in finished]
    model.eval(); tok.padding_side = "left"
    t0 = time.time()
    for s in range(0, len(todo), GEN_BATCH):
        rows = todo[s:s + GEN_BATCH]
        prompts = [prompt_text(frame.instruction_clean.iloc[r], frame.predicted_intent.iloc[r]) for r in rows]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None, top_p=None,
                             pad_token_id=tok.pad_token_id, eos_token_id=[tok.eos_token_id, EOT_ID])
        new = out[:, enc["input_ids"].shape[1]:].tolist()
        recs = []
        for r, seq in zip(rows, new):
            stop = next((k for k, t in enumerate(seq) if t in (EOT_ID, tok.eos_token_id)), None)
            seq_used = seq if stop is None else seq[:stop]
            recs.append({"row": r, "answer": tok.decode(seq_used, skip_special_tokens=True).strip(),
                         "new_tokens": len(seq_used), "hit_limit": stop is None})
        done = pd.concat([done, pd.DataFrame(recs)], ignore_index=True)
        done["row"] = done["row"].astype(int)
        done.to_csv(path, index=False)
        print(f"  {cache_name}: {len(done)}/{len(frame)} answers ({time.time() - t0:.0f}s)")
    tok.padding_side = "right"
    done["row"] = done["row"].astype(int)
    return done.sort_values("row").set_index("row").reindex(range(len(frame))), time.time() - t0

### 6a · Base model (zero-shot)

In [ ]:
model = load_base_model()
timing = {}
meta_path = os.path.join(CACHE_DIR, "ppl.json")
ppl = json.load(open(meta_path)) if os.path.exists(meta_path) else {}
if "base" not in ppl:
    ppl["base"] = reference_perplexity(model, llm_val); json.dump(ppl, open(meta_path, "w"))
print("Base model - perplexity of reference answers:", round(ppl["base"], 2))
base_out, timing["base_generation_s"] = generate_answers(model, llm_test, "answers_base.csv")
print(base_out.answer.iloc[0][:600])

### 6b · QLoRA fine-tuning
Skipped automatically if a finished adapter is already saved on Drive.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from tqdm.auto import tqdm

train_log_path = os.path.join(ART_DIR, "train_log.csv")
if os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    print("Found a trained adapter on Drive - loading it instead of retraining.")
    model = PeftModel.from_pretrained(model, ADAPTER_DIR)
    train_info = json.load(open(os.path.join(ADAPTER_DIR, "train_info.json")))
else:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                            gradient_checkpointing_kwargs={"use_reentrant": False})
    lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lora)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable parameters: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")

    tok.padding_side = "right"
    train_enc = [encode_example(q, a, i) for q, a, i in zip(llm_train.instruction_clean, llm_train.response_clean, llm_train.intent)]
    g = torch.Generator().manual_seed(SEED)
    loader = torch.utils.data.DataLoader(train_enc, batch_size=BATCH, shuffle=True, collate_fn=collate, generator=g)
    steps = math.ceil(len(loader) / GRAD_ACCUM) * EPOCHS
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    warm = max(1, int(0.05 * steps))   # linear warm-up, then cosine decay
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                                              else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))
    scaler = torch.amp.GradScaler("cuda")

    log, step, run_loss, t0 = [], 0, 0.0, time.time()
    model.train(); model.config.use_cache = False
    bar = tqdm(total=steps, desc="fine-tuning")
    for ep in range(EPOCHS):
        for i, b in enumerate(loader):
            b = {k: v.to("cuda") for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**b).loss / GRAD_ACCUM
            scaler.scale(loss).backward(); run_loss += float(loss)
            if (i + 1) % GRAD_ACCUM == 0 or i + 1 == len(loader):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
                step += 1
                log.append({"step": step, "epoch": ep + 1, "loss": run_loss, "lr": sched.get_last_lr()[0]})
                run_loss = 0.0; bar.update(1); bar.set_postfix(loss=round(log[-1]["loss"], 4))
    bar.close()
    model.config.use_cache = True
    train_info = {"train_minutes": round((time.time() - t0) / 60, 1), "optimizer_steps": step,
                  "trainable_params": trainable, "total_params": total}
    os.makedirs(ADAPTER_DIR, exist_ok=True)
    model.save_pretrained(ADAPTER_DIR)
    json.dump(train_info, open(os.path.join(ADAPTER_DIR, "train_info.json"), "w"))
    pd.DataFrame(log).to_csv(train_log_path, index=False)
    print("Training finished:", train_info)

if os.path.exists(train_log_path):
    pd.read_csv(train_log_path).plot(x="step", y="loss", title="Training loss", figsize=(8, 3)); plt.show()

### 6c · Fine-tuned model

In [ ]:
if "finetuned" not in ppl:
    ppl["finetuned"] = reference_perplexity(model, llm_val); json.dump(ppl, open(meta_path, "w"))
print("Perplexity of reference answers - base:", round(ppl["base"], 2), "| fine-tuned:", round(ppl["finetuned"], 2))
ft_out, timing["finetuned_generation_s"] = generate_answers(model, llm_test, "answers_finetuned.csv")
print(ft_out.answer.iloc[0][:600])

# free the GPU for the evaluation models
del model; import gc; gc.collect(); torch.cuda.empty_cache()

## 7 · Evaluation
Every answer from both models, plus the human-written reference answers as a benchmark, is scored with:

| Criterion | Metric | Better |
|---|---|---|
| Correctness | **ROUGE-L F1** and **BERTScore F1** (rescaled) against the reference answer | higher |
| Correctness (meaning) | cosine similarity of MiniLM embeddings, answer vs reference | higher |
| Relevance | cosine similarity of MiniLM embeddings, answer vs customer question | higher |
| Helpfulness | rule-based proxy: concrete actions and steps (0–1) | higher |
| Fluency | perplexity under an independent language model (GPT-2) | lower |
| Hallucination | % of answers stating a specific (order no., phone, link, price, time window) the customer never gave | lower |
| Grounding behaviour | % of answers that use a `{{placeholder}}` instead of inventing a value | higher |

Raw cosine similarity is reported, not rescaled to 0–1, so unrelated texts don't get inflated scores.
Differences are tested with a paired bootstrap 95% CI and a Wilcoxon signed-rank test (McNemar for yes/no
metrics).

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from scipy.stats import wilcoxon, binomtest
from transformers import AutoModelForCausalLM as _LM

answers = {"base": base_out.answer.fillna("").tolist(), "finetuned": ft_out.answer.fillna("").tolist(),
           "reference": llm_test.response_clean.tolist()}
questions = llm_test.instruction_clean.tolist()
refs = llm_test.response_clean.tolist()

ACTION_MARKERS = ["log in", "click", "visit", "navigate", "contact", "call", "select", "follow these steps",
                  "go to", "confirm", "provide", "sign in", "enter", "update", "check", "reach out"]
def helpfulness(t):
    tl = t.lower(); hits = sum(m in tl for m in ACTION_MARKERS)
    has_steps = bool(re.search(r"(^|\n)\s*(\d+[.)]|[-*•])\s", t))
    return min(1.0, 0.1 + 0.15 * hits + (0.3 if has_steps else 0))

rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
q_emb = st_model.encode(questions, normalize_embeddings=True, batch_size=64)
r_emb = st_model.encode(refs, normalize_embeddings=True, batch_size=64)

gpt2_tok = AutoTokenizer.from_pretrained("gpt2"); gpt2 = _LM.from_pretrained("gpt2").to("cuda").eval()
@torch.no_grad()
def log_ppl(text):
    ids = gpt2_tok(text, return_tensors="pt", truncation=True, max_length=512)["input_ids"].to("cuda")
    if ids.shape[1] < 2: return float("nan")
    return float(gpt2(ids, labels=ids).loss)

scores = {}
for name, ans in answers.items():
    a_emb = st_model.encode(ans, normalize_embeddings=True, batch_size=64)
    _, _, bf1 = bert_score(ans, refs, lang="en", rescale_with_baseline=True, verbose=False, batch_size=32)
    inv = [textprep.invented_specifics(a, q) for a, q in zip(ans, questions)]
    scores[name] = pd.DataFrame({
        "rougeL": [rs.score(r, a)["rougeL"].fmeasure for r, a in zip(refs, ans)],
        "bertscore": bf1.numpy(),
        "semantic_sim": (a_emb * r_emb).sum(1),
        "relevance": (a_emb * q_emb).sum(1),
        "helpfulness": [helpfulness(a) for a in ans],
        "log_perplexity": [log_ppl(a) for a in ans],
        "hallucination": [int(bool(x)) for x in inv],
        "uses_placeholder": [int(bool(textprep.extract_slots(a))) for a in ans],
        "words": [len(a.split()) for a in ans],
        "invented": [json.dumps(x) for x in inv],
    })
    print(name, "scored")

In [ ]:
METRICS = [  # key, label, higher_is_better, binary
    ("bertscore", "BERTScore F1", True, False), ("rougeL", "ROUGE-L F1", True, False),
    ("semantic_sim", "Semantic similarity to reference", True, False), ("relevance", "Relevance to question", True, False),
    ("helpfulness", "Helpfulness (proxy)", True, False), ("log_perplexity", "Fluency (GPT-2 perplexity)", False, False),
    ("hallucination", "Hallucination rate", False, True), ("uses_placeholder", "Uses placeholders", True, True),
]
rng = np.random.default_rng(SEED)
def compare(a, b, binary):
    d = np.asarray(b, float) - np.asarray(a, float); d = d[~np.isnan(d)]
    boots = rng.choice(d, (2000, len(d))).mean(1)
    if binary:
        a, b = np.asarray(a), np.asarray(b); n01 = int(((a == 1) & (b == 0)).sum()); n10 = int(((a == 0) & (b == 1)).sum())
        p = binomtest(min(n01, n10), n01 + n10, 0.5).pvalue if n01 + n10 else 1.0
    else:
        p = wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return float(d.mean()), [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))], float(p)

summary = {"n_test": len(llm_test), "metrics": [], "models": {
    "base": {"label": "Base Llama-3.2-3B-Instruct (zero-shot)"},
    "finetuned": {"label": "Fine-tuned Llama-3.2-3B-Instruct (QLoRA)"},
    "reference": {"label": "Reference answers (Bitext, human-written)"}}}
for key, label, hib, binary in METRICS:
    diff, ci, p = compare(scores["base"][key], scores["finetuned"][key], binary)
    better = "finetuned" if (diff > 0) == hib else "base"
    summary["metrics"].append({"key": key, "label": label, "higher_is_better": hib, "binary": binary,
        "base": float(np.nanmean(scores["base"][key])), "finetuned": float(np.nanmean(scores["finetuned"][key])),
        # reference vs itself is trivially perfect on the correctness metrics, so it is not reported there
        "reference": None if key in ("bertscore", "rougeL", "semantic_sim") else float(np.nanmean(scores["reference"][key])),
        "diff_ft_minus_base": diff, "ci95": ci, "p_value": p,
        "significant": bool(p < 0.05 and not (ci[0] <= 0 <= ci[1])), "better": better})
summary["reference_perplexity"] = ppl
summary["router_accuracy_on_test_sample"] = float((llm_test.predicted_intent == llm_test.intent).mean())
summary["pct_hit_token_limit"] = {"base": float(base_out.hit_limit.astype(bool).mean()), "finetuned": float(ft_out.hit_limit.astype(bool).mean())}
summary["hallucination_types"] = {}
for name in answers:
    c = {}
    for s in scores[name].invented:
        for k in json.loads(s): c[k] = c.get(k, 0) + 1
    summary["hallucination_types"][name] = c
sig = [m for m in summary["metrics"] if m["significant"]]
wins = {k: sum(m["better"] == k for m in sig) for k in ("base", "finetuned")}
bs = next(m for m in summary["metrics"] if m["key"] == "bertscore")
if wins["finetuned"] != wins["base"]:
    summary["best_model"] = "finetuned" if wins["finetuned"] > wins["base"] else "base"
else:  # tie on significant wins -> primary metric decides
    summary["best_model"] = "finetuned" if bs["finetuned"] >= bs["base"] else "base"
summary["significant_wins"] = wins
save_json(summary, "llm_summary.json")

gen = llm_test[["instruction_clean", "intent", "predicted_intent", "category", "response_clean"]].rename(
    columns={"instruction_clean": "question", "intent": "true_intent", "response_clean": "reference"}).copy()
for name in ("base", "finetuned"):
    gen[f"{name}_answer"] = answers[name]
for name in answers:
    for c in ["rougeL", "bertscore", "semantic_sim", "relevance", "helpfulness", "log_perplexity", "hallucination", "uses_placeholder", "words", "invented"]:
        gen[f"{name}_{c}"] = scores[name][c].values
gen.to_csv(os.path.join(ART_DIR, "generations.csv"), index=False)

per_int = []
for name in ("base", "finetuned"):
    t = gen.groupby("true_intent").agg(category=("category", "first"), bertscore=(f"{name}_bertscore", "mean"),
                                        rougeL=(f"{name}_rougeL", "mean"), hallucination=(f"{name}_hallucination", "mean")).reset_index()
    t["model"] = name; per_int.append(t)
pd.concat(per_int).to_csv(os.path.join(ART_DIR, "llm_per_intent.csv"), index=False)

pd.DataFrame([{"metric": m["label"], "base": round(m["base"], 4), "fine-tuned": round(m["finetuned"], 4),
               "reference": None if m["reference"] is None else round(m["reference"], 4), "diff": round(m["diff_ft_minus_base"], 4),
               "95% CI": f"[{m['ci95'][0]:.3f}, {m['ci95'][1]:.3f}]", "p": f"{m['p_value']:.2g}", "better": m["better"]}
              for m in summary["metrics"]])

## 8 · Interpretation (examples)

In [ ]:
gen["gain"] = gen.finetuned_bertscore - gen.base_bertscore
print("Best model:", summary["models"][summary["best_model"]]["label"], "| significant wins:", wins)
print("Hallucination types:", summary["hallucination_types"])
for _, r in gen.sort_values("gain", ascending=False).head(2).iterrows():
    print("\n" + "=" * 100 + f"\nQ: {r.question}  [{r.true_intent}]\n\nBASE:\n{r.base_answer[:700]}\n\nFINE-TUNED:\n{r.finetuned_answer[:700]}")

## 9 · Export for the Streamlit app
This saves `artifacts/` plus a `requirements.txt` that pins scikit-learn to the version used here (the saved
router models need the same version), then downloads **`group13_deploy.zip`**.

In [ ]:
import peft, numpy, zipfile
config = {
    "group": "Group 13", "course": "MSBA 610 Advanced Text Analytics",
    "created": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "model_requested": MODEL_ID, "model_used": MODEL_USED, "gpu": GPU_NAME,
    "n_train": len(llm_train), "n_val": len(llm_val), "n_test": len(llm_test),
    "max_len": MAX_LEN, "max_new_tokens": MAX_NEW_TOKENS,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT},
    "lr": LR, "epochs": EPOCHS, "effective_batch": BATCH * GRAD_ACCUM,
    "use_intent_hint": USE_INTENT_HINT, "system_prompt": SYSTEM_PROMPT,
    "best_router": BEST_ROUTER, "router_labels": ROUTERS,
    "train_info": train_info, "timing_seconds": timing,
    "versions": {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
                 "peft": peft.__version__, "scikit-learn": sklearn.__version__, "numpy": numpy.__version__},
}
save_json(config, "config.json")

req = f"""streamlit>=1.50
pandas
numpy>={numpy.__version__.split('.')[0]}.0
scikit-learn=={sklearn.__version__}
nltk
altair
"""
bundle = "/content/group13_deploy.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(ART_DIR)):
        z.write(os.path.join(ART_DIR, fn), f"artifacts/{fn}")
    z.writestr("requirements.txt", req)
shutil.copy(bundle, WORK_DIR)
print("Files:", sorted(os.listdir(ART_DIR)))
print("\nrequirements.txt:\n" + req)
try:
    from google.colab import files
    files.download(bundle)
except Exception:
    print("Download it from", bundle)